# Unity Catalog & Governance

**Module 6 | Fundamentals Training**

Unity Catalog is the central data governance system in Databricks — a three-level namespace (catalog.schema.table), fine-grained permissions, Row/Column Level Security, Data Lineage, and Delta Sharing. It is the foundation for a secure and auditable Lakehouse.

---


## Learning Objectives

After completing this module you will be able to:

- **Explain** Unity Catalog's three-level namespace and securable objects hierarchy
- **Grant and revoke** privileges using `GRANT`, `REVOKE`, and `DENY` SQL statements
- **Add comments and tags** to tables and columns for discoverability and documentation

## Setup

In [0]:
%run ../setup/00_setup

### Configuration
Define paths to data source files used throughout this notebook.

In [0]:
# Paths to data directories (subdirectories in DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
PRODUCTS_PATH = f"{DATASET_PATH}/products"

# Paths to specific files
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{PRODUCTS_PATH}/products.parquet"

## Unity Catalog Architecture

**Unity Catalog** is a unified governance solution for Databricks Lakehouse.

### Object Hierarchy:

```
Metastore (region-level)
 ↓
Catalog (database/domain)
 ↓
Schema (namespace)
 ↓
Securable Objects:
 - Tables / Views
 - Functions (UDF, stored procedures)
 - Volumes (files storage)
 - Models (ML models)
```

### Three-level namespace:
```sql
catalog.schema.table
```

Example:
```sql
main.sales.orders
dev.analytics.customer_metrics
prod.gold.daily_revenue
```

### Key Features:
- **Unified governance**: single platform for data, ML, BI
- **Fine-grained access control**: table, column, row level
- **Automatic lineage**: end-to-end data flow tracking
- **Audit logging**: who accessed what and when
- **Data discovery**: metadata search and tagging

---

<img src="../../assets/images/unity_catalog_namespace.svg" width="900">

### Setup and Basic Operations
Initialize Unity Catalog objects and verify the working environment.

### Creating User Groups
We create user groups for permission demonstration:
- `data_engineers`: Full access to Bronze/Silver schemas
- `data_analysts`: Read-only access to Gold

In [0]:
# Verification of created schemas
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").select("databaseName").collect()
schema_names = [row.databaseName for row in schemas]

**Active catalog and schema set**

We set the default working context - all subsequent operations will be executed in this catalog and schema unless a full path is specified.

In [0]:
# Verification of created schemas
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

## Data Preparation

Before we proceed to access management, we will load real data from the dataset/ directory that we will use in the Unity Catalog examples.

---

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders")

orders_df = spark.read.option("header", "true").option("inferSchema", "true").json(ORDERS_JSON)
orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")

customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv(CUSTOMERS_CSV)
customers_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.products")

products_df = spark.read.parquet(PRODUCTS_PARQUET)
products_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.products")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products"))

In [0]:
# Verification of orders record count
spark.sql(f"SELECT COUNT(*) as count FROM {CATALOG}.{BRONZE_SCHEMA}.orders").display()

### Demo: Comments and Tags

You can add descriptive comments to Unity Catalog tables and columns using SQL commands. This improves data discoverability and governance.

The cell below demonstrates how to add comments to a table and a specific column using Spark SQL.

In [0]:
# Add comments to table and columns
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders IS
    'Cleaned orders table with data quality validations applied'
""")

spark.sql(f"""
    COMMENT ON COLUMN {CATALOG}.{BRONZE_SCHEMA}.orders.customer_id IS
    'Customer identifier - PII data, access restricted'
""")

### Add tags to orders table

You can classify and manage tables in Unity Catalog using **tags** (key-value pairs). Tags help with data discovery, compliance, and governance (e.g., marking tables as PII, GDPR, or Sensitive).

Example: 
sql
ALTER TABLE retailhub_trainer.bronze.orders
 SET TAGS ('pii' = 'false', 'data_classification' = 'transactional', 'retention' = '7_years');

- `pii`: Indicates if table contains personally identifiable information.
- `data_classification`: Describes the type of data (e.g., transactional, reference).
- `retention`: Specifies data retention policy.

You need `APPLY TAG` privilege to add tags.

In [0]:
# Add tags to orders table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.orders 
    SET TAGS ('sensitivity' = 'high', 'domain' = 'sales')
""")

# Add tags to customer_id column
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.orders 
    ALTER COLUMN customer_id SET TAGS ('pii' = 'true')
""")

display(spark.createDataFrame([("Status", " Tags added to table and column")], ["Info", "Value"]))

## Querying Metadata and Tags

Exploring Unity Catalog metadata using system tables and information_schema to discover tags, comments, and permissions across your catalog.

---

In [0]:
# Find all columns marked as PII
pii_columns = spark.sql(f"""
    SELECT 
        catalog_name, 
        schema_name, 
        table_name, 
        column_name, 
        tag_value 
    FROM system.information_schema.column_tags
    WHERE tag_name = 'pii' AND tag_value = 'true'
      AND catalog_name = '{CATALOG}'
""")

display(pii_columns)

## Access Management: GRANT / REVOKE

Unity Catalog uses **SQL-standard GRANT / REVOKE** commands to control who can access what.

### Permission hierarchy (must grant each level top-down)

```
CATALOG → SCHEMA → TABLE / VIEW / FUNCTION
```

| Permission | Grants access to |
|---|---|
| `USE CATALOG` | Opens the catalog (required first) |
| `USE SCHEMA` | Opens a schema inside the catalog |
| `SELECT` | Read data from a table / view |
| `MODIFY` | INSERT, UPDATE, DELETE on a table |
| `CREATE TABLE` | Create new tables inside a schema |
| `EXECUTE` | Call a function / UDF |
| `ALL PRIVILEGES` | All of the above at once |

Grant to **users** (`user@company.com`) or **groups** (recommended for scale).

> **Important:** Always grant `USE CATALOG` before `USE SCHEMA`, and `USE SCHEMA` before `SELECT`.


<img src="../../assets/images/unity_catalog_permissions.svg" width="900">

**Setting permissions for data-analysts**

The `data-analysts` group received:
- **USE CATALOG**: Access to catalog
- **USE SCHEMA**: Access to Silver schema 
- **SELECT**: Read data from Silver schema

**Setup:** Create groups for demonstration purposes
> **Note:** This requires account admin privileges. If you don't have them, ensure these groups exist.

> **Important:** If the `analysts` group does not exist, any GRANT command will fail. You must go to **Settings** and create the group first. Creating groups requires admin privileges.



In [0]:
# Grant catalog access to data analysts
spark.sql(f"""
    GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`
""")

spark.sql(f"""
    GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

spark.sql(f"""
    GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

**Permissions for Data Analysts (Gold Layer):**

In [0]:
# GRANT for data-analysts on Gold schema
spark.sql(f"""
  GRANT USE SCHEMA ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

spark.sql(f"""
  GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

In [0]:
# Verify permissions on table
spark.sql(f"""
    SHOW GRANTS ON SCHEMA {CATALOG}.{GOLD_SCHEMA}
""").display()

**Granting permissions to RLS Views:**

In [0]:
# Revoke direct access to base table
spark.sql(f"""
    REVOKE SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders FROM `account users`
""")

---
## Data Lineage

**Lineage** in Unity Catalog is the automatic tracking of where data comes from and where it goes — with no additional tools to install.

### What does Unity Catalog track automatically?
- Where a table originates (source tables / files)
- What transformations were applied
- Which tables and columns depend on this table
- Who reads a given table and when

### How to browse Lineage:
1. Open **Catalog Explorer** in the left menu
2. Select any table in your catalog
3. Click the **Lineage** tab
4. Click **See lineage graph** — you will see an interactive graph

### What you will see in the graph:
```
  [customers.csv]   [orders.json]
        │                 │
        ▼                 ▼
  [bronze.customers] [bronze.orders]
        │                 │
        └────────┬────────┘
                 ▼
           [silver.orders_enriched]
                 │
                 ▼
           [gold.orders_summary]
                 │
                 ▼
           [AI/BI Dashboard]
```

### Column-level Lineage:
Unity Catalog tracks lineage not only at the table level, but **per column**. Clicking a specific column shows which source columns it originates from.

> **Practical use case:** When the business asks "Why did the KPI change?", Lineage lets you trace the full data path in seconds — without asking the data engineer.


In [0]:
## Lineage can also be queried via system tables (programmatically)

# Query system.access.table_lineage — Databricks collects this data automatically
try:
    display(
        spark.sql("""
            SELECT
                *
            FROM system.access.table_lineage
            ORDER BY event_time DESC
            LIMIT 20
        """)
    )
except Exception as e:
    print(f"Lineage system table requires appropriate permissions: {e}")
    print("Use Catalog Explorer GUI → Lineage tab as an alternative.")


---
## Delta Sharing

**Delta Sharing** is an open Databricks protocol for **securely sharing data externally** — without copying, migrating, or giving up control.

### The core problem it solves:

Without Delta Sharing:
> "Send me a CSV file" → data leaves the organisation, you lose control over who edits it, copies it, and how many versions are floating around.

With Delta Sharing:
> The recipient reads **from the source in real time** via a token; you can revoke that permission at any moment.

### Architecture:

```
  Provider (You)                   Recipient (Partner / Client)
  ──────────────                   ─────────────────────────────
  Unity Catalog                    Any Delta Sharing client:
  └─ SHARE my_share                  • Another Databricks workspace
       └─ TABLE gold.orders           • Apache Spark
       └─ TABLE gold.customers        • pandas (via REST API)
                                      • Power BI
                                      • Tableau
  Permissions controlled           No installation needed — just a
  by Unity Catalog                 token and endpoint URL
```

### Key concepts:

| Concept | Description |
|---------|-------------|
| **Share** | Container grouping tables to be shared |
| **Recipient** | Data consumer — identified by token or identity provider |
| **Provider** | Data owner (you) |
| `CREATE SHARE` | Creates a new share |
| `ALTER SHARE ... ADD TABLE` | Adds a table to a share |
| `CREATE RECIPIENT` | Creates a recipient and generates an activation token |
| `GRANT SELECT ON SHARE ... TO RECIPIENT` | Grants access |

### Minimal working example (SQL):

```sql
-- 1. Create a share
CREATE SHARE retailhub_partner_share
  COMMENT 'Sales data for partner X';

-- 2. Add tables
ALTER SHARE retailhub_partner_share ADD TABLE main.gold.orders_summary;
ALTER SHARE retailhub_partner_share ADD TABLE main.gold.product_catalog;

-- 3. Create a recipient (generates an activation link with a token)
CREATE RECIPIENT partner_company_x;

-- 4. Grant access
GRANT SELECT ON SHARE retailhub_partner_share TO RECIPIENT partner_company_x;

-- 5. Verify what is shared
SHOW ALL IN SHARE retailhub_partner_share;
```

### When to use Delta Sharing:
- Sharing data with external clients (B2B data products)
- Exchanging data between company divisions with separate workspaces
- Sharing data with vendors / analytical partners without giving them Databricks access
- Monetising data as a product (Data Marketplace)

> **Important:** Delta Sharing is an open-source protocol — the recipient **does not need** Databricks.
> They can use `pip install delta-sharing` or the Power BI connector.


---

## Summary

### Key Unity Catalog principles to remember:

| Concept | Rule |
|---------|------|
| **3-level namespace** | Every object lives at `catalog.schema.object` — always qualify! |
| **Metastore** | One per region; shared across all workspaces in the account |
| **GRANT / REVOKE** | Standard SQL syntax — works on Catalog, Schema, Table, Volume, View |
| **Least Privilege** | Grant `USE CATALOG` + `USE SCHEMA` + `SELECT` — minimum to read a table |
| **Row/Column Security** | Implement via Views, Dynamic Views, or Row Filters (Unity Catalog) |
| **Data Lineage** | Automatic — tracked in `system.access.table_lineage`; no config needed |
| **Tags** | Add metadata for compliance, PII marking, chargeback; queryable via system tables |
| **Delta Sharing** | Open protocol for sharing data externally without copying files |

### Access Management quick reference:
```sql
-- Read access to one table
GRANT USE CATALOG ON CATALOG my_catalog TO `analyst-group`;
GRANT USE SCHEMA   ON SCHEMA  my_catalog.gold TO `analyst-group`;
GRANT SELECT       ON TABLE   my_catalog.gold.sales TO `analyst-group`;

-- Revoke
REVOKE SELECT ON TABLE my_catalog.gold.sales FROM `analyst-group`;

-- Inspect
SHOW GRANTS ON TABLE my_catalog.gold.sales;
```

### One-line rules:
- **Unity Catalog is account-level** — one Metastore per region, multiple workspaces share it
- **Groups, not users** — always assign permissions to groups, never individual users
- **Views are your security layer** — use them for row/column filtering instead of sharing raw tables
- **Lineage is automatic** — just use Unity Catalog tables and Databricks tracks it

---
← [Orchestration & Lakeflow](05_orchestration_jobs.ipynb) | [AI/BI Dashboards & Genie →](07_aibi_dashboards.ipynb)